# HW05: Word Embeddings

Remember that these homework work as a completion grade. **You can <span style="color:red">not</span> skip one section this homework.**

**Training word2vec**

In this section, we train a word2vec model using gensim. We train the model on text8 (which consists of the first 90M characters of a Wikipedia dump from 2006 and is considered one of the benchmarks for evaluating language models).

In [1]:
import gensim.downloader as api

api.info("text8")

{'num_records': 1701,
 'record_format': 'list of str (tokens)',
 'file_size': 33182058,
 'reader_code': 'https://github.com/RaRe-Technologies/gensim-data/releases/download/text8/__init__.py',
 'license': 'not found',
 'description': 'First 100,000,000 bytes of plain text from Wikipedia. Used for testing purposes; see wiki-english-* for proper full Wikipedia datasets.',
 'checksum': '68799af40b6bda07dfa47a32612e5364',
 'file_name': 'text8.gz',
 'read_more': ['http://mattmahoney.net/dc/textdata.html'],
 'parts': 1}

In [2]:
dataset = api.load("text8")

In [3]:
from gensim.models import Word2Vec

##TODO train a word2vec model on this dataset which appear at least 10 times in the corpus
model = Word2Vec(sentences=dataset, vector_size=100, window=5, min_count=10, workers=4)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


**Word Similarities**

gensim models provide almost all the utility you might want to wish for to perform standard word similarity tasks. They are available in the .wv (wordvectors) attribute of the model, more details could be found [here](https://radimrehurek.com/gensim/models/keyedvectors.html).

In [4]:
model.wv

##TODO find the closest words to king
model.wv.most_similar("king")

[('prince', 0.7498952150344849),
 ('kings', 0.6993843913078308),
 ('vii', 0.6959946155548096),
 ('throne', 0.6908984780311584),
 ('queen', 0.6811890602111816),
 ('emperor', 0.6759054064750671),
 ('pharaoh', 0.6633204817771912),
 ('pope', 0.656085729598999),
 ('elector', 0.6554271578788757),
 ('regent', 0.6537782549858093)]

King is to man as woman is to X

In [5]:
##TODO find the closest word for the vector "woman" + "king" - "man"
model.wv.most_similar(positive=["woman", "king"], negative=["man"])

[('queen', 0.6521567702293396),
 ('empress', 0.6455972790718079),
 ('isabella', 0.636926531791687),
 ('prince', 0.61979079246521),
 ('throne', 0.6183339357376099),
 ('princess', 0.6076090931892395),
 ('elizabeth', 0.6029301881790161),
 ('matilda', 0.5983853936195374),
 ('daughter', 0.5884416103363037),
 ('sigismund', 0.5808001756668091)]

**Evaluate Word Similarities** 

One common way to evaluate word2vec models are word analogy tasks. Let's check how good our model is on one of those. We consider the [WordSim353](https://gabrilovich.com/resources/data/wordsim353/wordsim353.html) benchmark, the task is to determine how similar two words are.

In [7]:
!curl -L -O https://github.com/CallMeJiaGu/WordSimilarityAnalogyData/raw/refs/heads/master/ws353simrel.tar.gz
!tar xf ws353simrel.tar.gz

path = "wordsim353_sim_rel/wordsim_similarity_goldstandard.txt"

def load_data(path):
    X, y = [], []
    with open(path) as f:
        for line in f:
            line = line.strip().split("\t")
            X.append((line[0], line[1])) # each entry in x contains two words, e.g. X[0] = (tiger, cat)
            y.append(float(line[-1])) # each entry in y is the annotation how similar two words are, e.g. Y[0] = 7.35
    return X, y

X, y = load_data(path)
print (X[:3], y[:3])

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  5460  100  5460    0     0  10277      0 --:--:-- --:--:-- --:--:-- 10277
[('tiger', 'cat'), ('tiger', 'tiger'), ('plane', 'car')] [7.35, 10.0, 5.77]


In [8]:
##TODO compute how similar the pairs in the WordSim353 are according to our model
# if a word is not present in our model, we assign similarity 0 for the respective text pair
predictions = []
for w1, w2 in X:
    if w1 in model.wv and w2 in model.wv:
        predictions.append(model.wv.similarity(w1, w2))
    else:
        predictions.append(0)
predictions[:3]

[np.float32(0.58840936), np.float32(1.0), np.float32(0.44332168)]

In [9]:
from scipy.stats import spearmanr

##TODO compute spearman's rank correlation between our prediction and the human annotations
correlation, pvalue = spearmanr(predictions, y)
print(f"Spearman correlation: {correlation:.4f}, p-value: {pvalue:.4f}")

Spearman correlation: 0.6567, p-value: 0.0000


In [10]:
import spacy
en = spacy.load('en_core_web_sm')

##TODO compute word similarities in the WordSim353 dataset using spaCy word embeddings
spacy_predictions = []
for w1, w2 in X:
    t1, t2 = en(w1)[0], en(w2)[0]
    if t1.has_vector and t2.has_vector:
        spacy_predictions.append(t1.similarity(t2))
    else:
        spacy_predictions.append(0)

##TODO compute spearman's rank correlation between these similarities and the human annotations
# Don't worry if results are not too convincing for this experiment
spacy_corr, spacy_pvalue = spearmanr(spacy_predictions, y)
print(f"spaCy Spearman correlation: {spacy_corr:.4f}, p-value: {spacy_pvalue:.4f}")

/var/folders/s0/qj_3vfv57l5bknx99kmph6sc0000gn/T/ipykernel_61950/3622073900.py:9: UserWarning: [W007] The model you're using has no word vectors loaded, so the result of the Token.similarity method will be based on the tagger, parser and NER, which may not give useful similarity judgements. This may happen if you're using one of the small models, e.g. `en_core_web_sm`, which don't ship with word vectors and only use context-sensitive tensors. You can always add your own word vectors, or use one of the larger models instead if available.
  spacy_predictions.append(t1.similarity(t2))


spaCy Spearman correlation: 0.0492, p-value: 0.4856


**PyTorch Embeddings**

In [11]:
#Import the AG news dataset (same as hw01)
#Download them from here 
# !wget https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/train.csv

import pandas as pd
import nltk
df = pd.read_csv('train.csv')

df.columns = ["label", "title", "lead"]
label_map = {1:"world", 2:"sport", 3:"business", 4:"sci/tech"}
def replace_label(x):
	return label_map[x]
df["label"] = df["label"].apply(replace_label) 
df["text"] = df["title"] + " " + df["lead"]
df = df.sample(n=10000) # # only use 10K datapoints
df.head()

,label,title,lead,text
29680,business,"Bankruptcy looming at Delta, auditor says",Delta Air Lines said Wednesday it has revised ...,"Bankruptcy looming at Delta, auditor says Delt..."
65998,sci/tech,S. Korea Backs UN Meeting on Stem Cell Research,South Korea said on Tuesday it asked the Unite...,S. Korea Backs UN Meeting on Stem Cell Researc...
73468,world,Fugitive Karadzic's Novel Best Seller (AP),AP - A novel by Bosnian Serb wartime leader Ra...,Fugitive Karadzic's Novel Best Seller (AP) AP ...
51751,business,Rupert blinks as the ISS giant awakes,Rupert Murdoch #39;s brazen attempt to relocat...,Rupert blinks as the ISS giant awakes Rupert M...
1359,world,Hostile Takeover,The battle moves to the political arena as Tai...,Hostile Takeover The battle moves to the polit...


In [12]:
vocab = 200
##TODO tokenize the text, only keep 200 most frequent words
from collections import Counter

tokenized = df["text"].str.lower().apply(nltk.word_tokenize)

word_counts = Counter(word for tokens in tokenized for word in tokens)
top_words = {word for word, _ in word_counts.most_common(vocab)}

word2idx = {word: idx + 1 for idx, (word, _) in enumerate(word_counts.most_common(vocab))}
word2idx["<unk>"] = 0

tokenized = tokenized.apply(lambda tokens: [w if w in top_words else "<unk>" for w in tokens])
tokenized.head()

29680    [<unk>, <unk>, at, <unk>, ,, <unk>, says, <unk...
65998    [<unk>, <unk>, <unk>, <unk>, <unk>, on, <unk>,...
73468    [<unk>, <unk>, 's, <unk>, <unk>, <unk>, (, ap,...
51751    [<unk>, <unk>, as, the, <unk>, <unk>, <unk>, <...
1359     [<unk>, <unk>, the, <unk>, <unk>, to, the, <un...
Name: text, dtype: object

In [13]:
length = 100
#TODO create a one_hot representation for each word and truncate/pad the sequences such that they are all of the same length (here we use 100)
import numpy as np

def encode_and_pad(tokens, word2idx, length):
    indices = [word2idx.get(w, 0) for w in tokens]
    indices = indices[:length]                        # truncate
    indices += [0] * (length - len(indices))         # pad with <unk> index
    return indices

sequences = np.array([encode_and_pad(tokens, word2idx, length) for tokens in tokenized])

vocab_size = len(word2idx)
one_hot = np.zeros((len(sequences), length, vocab_size), dtype=np.float32)
for i, seq in enumerate(sequences):
    for j, idx in enumerate(seq):
        one_hot[i, j, idx] = 1.0

print(f"sequences shape: {sequences.shape}")   # (10000, 100)
print(f"one_hot shape:   {one_hot.shape}")     # (10000, 100, 201)

sequences shape: (10000, 100)
one_hot shape:   (10000, 100, 201)


In [14]:
##TODO create your torch embedding like we did in notebook 5! (hint: predicting labels: world, sport, business, and sci/tech)
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# encode labels
label2idx = {"world": 0, "sport": 1, "business": 2, "sci/tech": 3}
labels = np.array(df["label"].map(label2idx))

X_tensor = torch.tensor(sequences, dtype=torch.long)
y_tensor = torch.tensor(labels, dtype=torch.long)

dataset = TensorDataset(X_tensor, y_tensor)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

# model: embedding -> mean pool -> classifier
class EmbeddingClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.fc = nn.Linear(embedding_dim, num_classes)

    def forward(self, x):
        embedded = self.embedding(x)       # (batch, seq_len, emb_dim)
        pooled = embedded.mean(dim=1)      # (batch, emb_dim)
        return self.fc(pooled)

model_clf = EmbeddingClassifier(vocab_size=vocab_size, embedding_dim=64, num_classes=4)
optimizer = torch.optim.Adam(model_clf.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# training loop
for epoch in range(5):
    total_loss, correct = 0, 0
    for X_batch, y_batch in loader:
        optimizer.zero_grad()
        logits = model_clf(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y_batch)
        correct += (logits.argmax(dim=1) == y_batch).sum().item()
    print(f"Epoch {epoch+1}: loss={total_loss/len(dataset):.4f}, acc={correct/len(dataset):.4f}")

Epoch 1: loss=1.3697, acc=0.3428
Epoch 2: loss=1.3201, acc=0.4940
Epoch 3: loss=1.2452, acc=0.5524
Epoch 4: loss=1.1569, acc=0.5921
Epoch 5: loss=1.0722, acc=0.6192
